<a href="https://colab.research.google.com/github/shagun30107-bit/csot-ml-astronomy/blob/main/WEEK2(PART2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cpu


In [46]:
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2" / "images"
DATA_ROOT = Path("galaxy_data")
LABELS_URL = "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"


!pip install -q kaggle
# Kaggle authentication
os.environ["KAGGLE_API_TOKEN"] = "KGAT_3f3d786e5f471c9718d9cc3a6e481eb9"

# Download Galaxy Zoo 2 dataset
!kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images -p galaxy_raw

# Extract dataset
!unzip -q -o galaxy_raw/galaxy-zoo-2-images.zip -d {RAW_ROOT}

# !unzip -q -o {RAW_ROOT}/images_gz2.zip -d {IMAGES_DIR}

# Download Hart et al. labels
!wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz {LABELS_URL}
!gunzip -f {RAW_ROOT}/gz2_hart16.csv.gz


Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
galaxy-zoo-2-images.zip: Skipping, found more recently modified local copy (use --force to force download)


In [47]:
print("RAW_ROOT contents:", sorted(p.name for p in RAW_ROOT.iterdir()))
jpg_count = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print(f"Flat JPG count in {IMAGES_DIR}: {jpg_count:,}")

mapping_head = pd.read_csv(RAW_ROOT / "gz2_filename_mapping.csv", nrows=3)

RAW_ROOT contents: ['galaxy-zoo-2-images.zip', 'gz2_filename_mapping.csv', 'gz2_hart16.csv', 'images_gz2']
Flat JPG count in galaxy_raw/images_gz2/images: 243,434


In [48]:
def high_level_label(gz2_class: str):
    """Collapse detailed GZ2 morphology codes to a few training buckets."""
    if not gz2_class or gz2_class == "A":
        return None  # artifact / ambiguous — skip
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None


def build_imagefolder_layout(
    images_dir,
    mapping_csv,
    labels_csv,
    out_root,
    per_class=200,
    seed=42,
):
    """Symlink a balanced subset into out_root// for ImageFolder."""
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv).rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    df = df.dropna(subset=["label"])

    images_dir = Path(images_dir)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    counts = {}
    for label in sorted(df["label"].unique()):
        class_dir = out_root / label
        class_dir.mkdir(exist_ok=True)
        rows = df[df["label"] == label]
        if len(rows) > per_class:
            rows = rows.sample(n=per_class, random_state=seed)
        linked = 0
        for _, row in rows.iterrows():
            src = images_dir / f"{int(row.asset_id)}.jpg"
            dst = class_dir / f"{int(row.asset_id)}.jpg"
            if src.exists() and not dst.exists():
                os.symlink(src.resolve(), dst)
                linked += 1
        counts[label] = linked
    return counts

PER_CLASS = 200  # balanced subset — fast on Colab; increase once the pipeline works
counts = build_imagefolder_layout(
    IMAGES_DIR,
    RAW_ROOT / "gz2_filename_mapping.csv",
    RAW_ROOT / "gz2_hart16.csv",
    DATA_ROOT,
    per_class=PER_CLASS,
)
print("Symlinked per class:", counts)
print("DATA_ROOT classes:", sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir()))

Symlinked per class: {'elliptical': 0, 'spiral': 0, 'spiral_barred': 0}
DATA_ROOT classes: ['elliptical', 'spiral', 'spiral_barred']


In [49]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

In [50]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split

dataset = ImageFolder(root=DATA_ROOT, transform=transform)

train_size = int(0.7 * len(dataset))
val_size   = int(0.15 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset,
    [train_size, val_size, test_size]
)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)



In [51]:
num_classes = len(train_ds.dataset.classes)
print("Classes      :", train_ds.dataset.classes)
print("Class_to_Idx :", train_ds.dataset.class_to_idx)
print("Num_Classes  :", num_classes)

Classes      : ['elliptical', 'spiral', 'spiral_barred']
Class_to_Idx : {'elliptical': 0, 'spiral': 1, 'spiral_barred': 2}
Num_Classes  : 3


# STEP-1

In [52]:
import torch
import torch.nn as nn

class GalaxyMLP(nn.Module):
    def __init__(self, in_features=3*64*64, hidden=128, num_classes=3):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(in_features, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# STEP-2

In [53]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GalaxyMLP(num_classes=num_classes ).to(device)
batch = torch.randn(32, 3, 64, 64)
batch = batch.to(device)        # input must match the model's device
logits = model(batch)
print(logits.device)            # cuda:0

cpu


# STEP-3

In [54]:
print(model)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")

GalaxyMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=12288, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)
Total parameters     : 1,573,379
Trainable parameters : 1,573,379


# STEP-4

In [55]:
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
logits = model(images)
print(logits.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([32, 3])


# STEP-5

In [56]:
import torch.optim as optim
import math

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# STEP-6

In [57]:
loss = criterion(logits, labels)
log = math.log(num_classes)
print(f"loss : {loss.item()}")
print(f"log: {log}")


loss : 1.0083256959915161
log: 1.0986122886681098


# STEP-7

In [58]:
model.train()
print("loss before step:", loss.item())

optimizer.zero_grad()
loss.backward()
optimizer.step()

logits2 = model(images)
loss_after = criterion(logits2, labels)
print("loss after step:", loss_after.item())


loss before step: 1.0083256959915161
loss after step: 3.6216325759887695


# STEP-8

In [59]:
for i in (64, 128, 256, 512):
    m = GalaxyMLP(hidden=i, num_classes=num_classes)
    n = sum(p.numel() for p in m.parameters())
    print(f"hidden={i:4d}  =  {n:,} parameters")

hidden=  64  =  786,691 parameters
hidden= 128  =  1,573,379 parameters
hidden= 256  =  3,146,755 parameters
hidden= 512  =  6,293,507 parameters


# REFLECTION


1.  Difference b/w them around 0.6 Not too much near-
At the start of training, the model is essentially guessing. With C possible classes, each class gets about 1/C probability, so the expected cross-entropy loss is around ln(C).

2.  This result is based on a single batch. During real training, the model learnes from all batches over many epochs and is evaluated using unseen data.

3. The first layer is huge bcoz it connects all 12288 image inputs to each hidden unit creating many weights... CNN are more efficient because they reuse small filters across the image, reducing the number of parameters.

